# Lab 03: API Testing - Sandboxes and Test Cards

**Lab**: 03-api-testing  
**Duration**: ~5 minutes  
**Prerequisites**: Completed `01_introduction.ipynb`

## Learning Objectives

By the end of this notebook, you will:
- Create payments using test cards
- Simulate successful and declined transactions
- Implement proper error handling for payment failures
- Use PaymentMethod identifiers in code

---

## Setup

Let's verify our environment and connect to Stripe.

<!-- PRESENTER: Ensure attendees have their .env file configured -->

In [ ]:
# Environment setup
import os
from dotenv import load_dotenv
import stripe

load_dotenv()
stripe.api_key = os.environ.get('STRIPE_SECRET_KEY')

# Verify connection
try:
    account = stripe.Account.retrieve()
    print(f"Connected to Stripe account: {account.id}")
    print(f"Mode: {'Sandbox' if 'test' in stripe.api_key else 'LIVE - BE CAREFUL!'}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key. Check your .env file.")

## Test Cards Reference

Stripe provides special card numbers for testing. These cards work ONLY in sandboxes (with `sk_test_` keys).

### Successful Payment Cards

| Brand | Number | PaymentMethod |
|-------|--------|---------------|
| Visa | 4242424242424242 | `pm_card_visa` |
| Visa (debit) | 4000056655665556 | `pm_card_visa_debit` |
| Mastercard | 5555555555554444 | `pm_card_mastercard` |
| Mastercard (debit) | 5200828282828210 | `pm_card_mastercard_debit` |
| American Express | 378282246310005 | `pm_card_amex` |

### For All Test Cards
- **Expiration**: Any future date (e.g., 12/34)
- **CVC**: Any 3 digits (4 for Amex)
- **ZIP**: Any 5 digits

<!-- PRESENTER: Note that these are the most commonly used cards -->

## Section 1: Creating a Successful Payment

Let's create a PaymentIntent using a test card. In production, you would collect card details through Stripe Elements or Checkout. For testing, we use `PaymentMethod` identifiers directly.

### PaymentMethod vs Card Numbers

In code, use `PaymentMethod` identifiers like `pm_card_visa` instead of raw card numbers. This is:
- More secure (no raw card data in code)
- PCI compliant
- The pattern you'll use in production

In [ ]:
# Create a successful payment with a test Visa card
payment_intent = stripe.PaymentIntent.create(
    amount=2000,  # $20.00 in cents
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,  # Immediately confirm the payment
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

print(f"Payment ID: {payment_intent.id}")
print(f"Amount: ${payment_intent.amount / 100:.2f} {payment_intent.currency.upper()}")
print(f"Status: {payment_intent.status}")

# Expected output:
# Payment ID: pi_xxx
# Amount: $20.00 USD
# Status: succeeded

### Checkpoint

At this point, you should see:
- A PaymentIntent ID starting with `pi_`
- Status: `succeeded`
- Amount: $20.00 USD

**Dashboard**: Navigate to [Payments](https://dashboard.stripe.com/test/payments) to see your test payment.

<!-- PRESENTER: Show the payment in the Dashboard -->

## Section 2: Simulating Declined Payments

Real-world payments fail for many reasons. Test cards let you simulate these scenarios.

### Common Decline Reasons

| Scenario | Card Number | PaymentMethod |
|----------|-------------|---------------|
| Generic decline | 4000000000000002 | `pm_card_visa_chargeDeclined` |
| Insufficient funds | 4000000000009995 | `pm_card_visa_chargeDeclinedInsufficientFunds` |
| Lost card | 4000000000009987 | `pm_card_visa_chargeDeclinedLostCard` |
| Stolen card | 4000000000009979 | `pm_card_visa_chargeDeclinedStolenCard` |
| Expired card | 4000000000000069 | `pm_card_chargeDeclinedExpiredCard` |
| Incorrect CVC | 4000000000000127 | `pm_card_chargeDeclinedIncorrectCvc` |

In [ ]:
# This will fail! Let's see what happens without error handling
try:
    payment_intent = stripe.PaymentIntent.create(
        amount=1500,  # $15.00
        currency="usd",
        payment_method="pm_card_visa_chargeDeclined",
        confirm=True,
        automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
    )
except stripe.error.CardError as e:
    print(f"Payment declined!")
    print(f"Error code: {e.code}")
    print(f"Decline code: {e.error.decline_code}")
    print(f"Message: {e.user_message}")

## Section 3: Proper Error Handling

Your integration MUST handle payment failures gracefully. Here's the recommended pattern:

### Error Types

| Error Type | When It Occurs | User Action |
|------------|----------------|-------------|
| `CardError` | Card declined, invalid | Ask for different card |
| `InvalidRequestError` | Invalid API parameters | Fix your code |
| `AuthenticationError` | Bad API key | Check configuration |
| `RateLimitError` | Too many requests | Retry with backoff |
| `StripeError` | General Stripe error | Retry or contact support |

In [ ]:
def create_payment(amount_cents: int, currency: str, payment_method: str) -> dict:
    """
    Create a payment with comprehensive error handling.
    
    Args:
        amount_cents: Amount in smallest currency unit (cents for USD)
        currency: Three-letter currency code
        payment_method: PaymentMethod ID (e.g., pm_card_visa)
    
    Returns:
        dict with 'success', 'payment_intent' or 'error' keys
    """
    try:
        payment_intent = stripe.PaymentIntent.create(
            amount=amount_cents,
            currency=currency,
            payment_method=payment_method,
            confirm=True,
            automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
        )
        return {
            "success": True,
            "payment_intent": payment_intent
        }
    
    except stripe.error.CardError as e:
        # Card was declined
        return {
            "success": False,
            "error": {
                "type": "card_error",
                "code": e.code,
                "decline_code": getattr(e.error, 'decline_code', None),
                "message": e.user_message
            }
        }
    
    except stripe.error.InvalidRequestError as e:
        # Invalid parameters
        return {
            "success": False,
            "error": {
                "type": "invalid_request",
                "message": str(e)
            }
        }
    
    except stripe.error.AuthenticationError as e:
        # API key issue
        return {
            "success": False,
            "error": {
                "type": "authentication_error",
                "message": "Invalid API key"
            }
        }
    
    except stripe.error.StripeError as e:
        # Generic Stripe error
        return {
            "success": False,
            "error": {
                "type": "stripe_error",
                "message": str(e)
            }
        }

print("Payment helper function defined!")

In [ ]:
# Test different payment scenarios

test_cases = [
    ("Successful Visa", "pm_card_visa", 1000),
    ("Declined (Generic)", "pm_card_visa_chargeDeclined", 1500),
    ("Insufficient Funds", "pm_card_visa_chargeDeclinedInsufficientFunds", 2000),
    ("Successful Mastercard", "pm_card_mastercard", 2500),
]

print("Testing Payment Scenarios")
print("=" * 60)

for name, pm, amount in test_cases:
    result = create_payment(amount, "usd", pm)
    
    if result["success"]:
        pi = result["payment_intent"]
        print(f"\n{name}:")
        print(f"  Status: SUCCESS")
        print(f"  Payment ID: {pi.id}")
        print(f"  Amount: ${pi.amount / 100:.2f}")
    else:
        error = result["error"]
        print(f"\n{name}:")
        print(f"  Status: FAILED")
        print(f"  Error Type: {error['type']}")
        print(f"  Message: {error['message']}")
        if error.get('decline_code'):
            print(f"  Decline Code: {error['decline_code']}")

### Checkpoint

You should see:
- 2 successful payments (Visa and Mastercard)
- 2 declined payments with specific decline codes

**Dashboard**: Check [Payments](https://dashboard.stripe.com/test/payments) - you'll see both successful and failed payment attempts.

<!-- PRESENTER: Show the payments list including failed attempts -->

## Best Practices for Test Cards

### 1. Use PaymentMethod Identifiers in Code

```python
# Good - PCI compliant
payment_method = "pm_card_visa"
```

### 2. Test All Failure Scenarios

Your integration should handle:
- Generic declines
- Insufficient funds
- Expired cards
- Invalid CVC
- Network errors

### 3. Provide User-Friendly Messages

```python
decline_messages = {
    "generic_decline": "Your card was declined. Please try another card.",
    "insufficient_funds": "Insufficient funds. Please try another card.",
    "expired_card": "Your card has expired. Please update your card.",
    "incorrect_cvc": "Invalid security code. Please check and try again.",
}
```

### 4. Log Errors for Debugging

Always log the full error for debugging while showing friendly messages to users.

## Summary

In this notebook, you learned:

- **Test cards** simulate real payment scenarios without moving money
- **PaymentMethod identifiers** (`pm_card_visa`) are preferred over raw card numbers
- **Error handling** is critical - always catch `CardError` and other exceptions
- **Decline codes** help you provide specific feedback to users

## Key Test Cards to Remember

| Scenario | PaymentMethod |
|----------|---------------|
| Success | `pm_card_visa` |
| Decline | `pm_card_visa_chargeDeclined` |
| Insufficient funds | `pm_card_visa_chargeDeclinedInsufficientFunds` |

## Next Steps

Continue to `03_test_clocks.ipynb` to learn how to simulate time for subscriptions!